## RAG Foundation

#### Flow:
- Load documents
- chunking
- embedding
- vector store
- retrieval
- Querying
- Reranking

#### Load documents

In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

directory_path = "data/pdf_files/"

loader = DirectoryLoader(
    path = directory_path,
    glob = "**/*.pdf",
    loader_cls = PyPDFLoader  
)

documents = loader.load()
print(f"Number of documents loaded: {len(documents)}")

f:\WeCloudData\AI\AgenticAI\langchain_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of documents loaded: 17


##### PII removal

In [16]:
# # Installing NLP Engine

# from presidio_analyzer.nlp_engine import NlpEngineProvider

# configuration = {
#     "nlp_engine_name": "spacy",
#     "models": [
#         {
#             "lang_code": "en",
#             "model_name": "en_core_web_sm"
#         }
#     ]
# }

# provider = NlpEngineProvider(nlp_configuration=configuration)
# nlp_engine = provider.create_engine()


# # test
# import spacy
# spacy.load("en_core_web_sm")

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
import traceback

configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {
            "lang_code": "en",
            "model_name": "en_core_web_sm"
        }
    ]
}

provider = NlpEngineProvider(
    nlp_configuration=configuration
)

nlp_engine = provider.create_engine()

analyzer = AnalyzerEngine(
    nlp_engine=nlp_engine
)

In [ ]:
text = """ 
Applicant John Doe (SSN: 999-00-1234) submitted his final mortgage inquiry on March 14, 2026. 
For correspondence, he used the email john.doe@example.com and the mobile 
number +1 (555) 019-8374. His current residential address is 456 Maple Drive, 
Springfield, IL 62704, and his driver’s license number is DL-8837492-X.
"""

# Step 1: Detect PII
analyzer_results = analyzer.analyze(
    text=text,
    language="en"
)

for result in analyzer_results:
    print(result)

type: EMAIL_ADDRESS, start: 135, end: 155, score: 1.0
type: PERSON, start: 12, end: 20, score: 0.85
type: ORGANIZATION, start: 22, end: 25, score: 0.85
type: DATE_TIME, start: 80, end: 94, score: 0.85
type: PERSON, start: 237, end: 248, score: 0.85
type: LOCATION, start: 251, end: 262, score: 0.85
type: ORGANIZATION, start: 264, end: 272, score: 0.85
type: PERSON, start: 309, end: 323, score: 0.85
type: PHONE_NUMBER, start: 182, end: 196, score: 0.75
type: URL, start: 135, end: 142, score: 0.5
type: URL, start: 144, end: 155, score: 0.5
type: US_DRIVER_LICENSE, start: 312, end: 319, score: 0.4


In [19]:
# Step 2: Anonymize PII, and mask/replace

from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

anonymizer = AnonymizerEngine()

anonymized_result = anonymizer.anonymize(
    text = text, 
    analyzer_results = analyzer_results,
    operators = {
        "PERSON": OperatorConfig(
            "replace",
            {"new_value": "[NAME]"} # default: <PERSON>
        ),
        "PHONE_NUMBER": OperatorConfig(
            "mask",
            {
                "masking_char": "*",
                "chars_to_mask": 6,
                "from_end": True
            }
        )
    }  
)


print(text)

print(anonymized_result)

 
Applicant John Doe (SSN: 999-00-1234) submitted his final mortgage inquiry on March 14, 2026. 
For correspondence, he used the email john.doe@example.com and the mobile 
number +1 (555) 019-8374. His current residential address is 456 Maple Drive, 
Springfield, IL 62704, and his driver’s license number is DL-8837492-X.

text:  
Applicant [NAME] (<ORGANIZATION>: 999-00-1234) submitted his final mortgage inquiry on <DATE_TIME>. 
For correspondence, he used the email <EMAIL_ADDRESS> and the mobile 
number +1 (555) 01******. His current residential address is 456 [NAME], 
<LOCATION>, <ORGANIZATION>, and his driver’s license number is [NAME]
items:
[
    {'start': 310, 'end': 316, 'entity_type': 'PERSON', 'text': '[NAME]', 'operator': 'replace'},
    {'start': 259, 'end': 273, 'entity_type': 'ORGANIZATION', 'text': '<ORGANIZATION>', 'operator': 'replace'},
    {'start': 247, 'end': 257, 'entity_type': 'LOCATION', 'text': '<LOCATION>', 'operator': 'replace'},
    {'start': 238, 'end': 244,

In [20]:
# Final analymized result
print(anonymized_result.text)

 
Applicant [NAME] (<ORGANIZATION>: 999-00-1234) submitted his final mortgage inquiry on <DATE_TIME>. 
For correspondence, he used the email <EMAIL_ADDRESS> and the mobile 
number +1 (555) 01******. His current residential address is 456 [NAME], 
<LOCATION>, <ORGANIZATION>, and his driver’s license number is [NAME]


#### Chunking

#### Embedding

#### Vector Store

#### Retrieval

#### Querying

#### Reranking